# 06 · Learning dynamics

**Paper:** "Superposition and Learning Dynamics", pp. 26–28.
**Experiment:** E05 `learning_dynamics`.

**Learning goals**

1. Track a model's geometry *during* training with the feature dimensionality Dᵢ (p. 18).
2. See the paper's "energy level" jumps: features hop between dimensionalities, and the loss
   drops when they do (p. 26).
3. Watch a 3D model learn through a sequence of geometric transformations: sets bunch up,
   expand into triangles, then rotate into an octahedron (p. 27).
4. Understand why training can stall in a nearly-right configuration for a long time.

The first run uses the `quick` profile: about 5 minutes for part (a) and 3 for part (b) on a
laptop CPU (`device="auto"` uses a GPU when available). If you already ran
`uv run superposition run learning_dynamics`, the notebook reuses that run.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display

from superposition import lab
from superposition.experiments.runner import run_experiment
from superposition.runs import list_runs

pd.set_option("display.max_colwidth", 160)


REUSE = True  # reuse your latest quick-profile run if there is one; False always re-runs


def quick_run(name: str):
    """Your latest complete quick-profile run of ``name``, or a fresh quick run."""
    if REUSE:
        for run in reversed(list_runs(name)):
            if yaml.safe_load(run.config_path.read_text())["profile"] == "quick":
                print(f"reusing run {run.run_id}")
                return run
    return run_experiment(name, "quick", device="auto")


def criteria(run) -> pd.DataFrame:
    doc = run.read_acceptance()
    print("overall:", doc["overall"], f"({doc['n_passed']}/{doc['n_graded']} graded criteria)")
    return pd.DataFrame(doc["criteria"])[["group", "id", "status", "value"]]

## 1. Energy-level jumps (p. 26)

Setup: the uniform model of p. 17, n = 400 features of equal importance in m = 30 dimensions,
at 1/(1 − S) = 3. On p. 17 this sparsity sits on the D* = 1/2 plateau: the model stores
features as **antipodal pairs** (digons, Dᵢ = 1/2) and ignores the rest (Dᵢ = 0).

**Predict** before running: how many features should end up represented? What Σᵢ Dᵢ do you
expect if every dimension holds one digon? Will every feature's Dᵢ move smoothly, or jump?

In [ ]:
run = quick_run("learning_dynamics")
criteria(run)

In [ ]:
display(Image(filename=str(run.figures_dir / "energy_levels.png")))

Blue lines end at 1/2, gray ones at 0; orange lines are the late jumpers the detector found (a
jump from a level held for 5,000 steps, by more than 0.15 within 2,000 steps, with a loss drop
at least three times faster than during the plateau and larger than its fluctuations).

Let's count features at each level at a few moments of training (restart 0):

In [ ]:
energy = run.load_checkpoint("energy_levels")
steps, dims = energy["steps"], energy["dimensionality"]
rows = []
for target in [0, 1_000, 3_000, 10_000, 30_000, int(steps[-1])]:
    t = int(np.searchsorted(steps, target))
    d = dims[t, 0]
    rows.append(
        {
            "step": int(steps[t]),
            "Dᵢ ≈ 1/2": int((np.abs(d - 0.5) <= 0.05).sum()),
            "Dᵢ ≈ 0": int((d <= 0.05).sum()),
            "elsewhere": int(((d > 0.05) & (np.abs(d - 0.5) > 0.05)).sum()),
            "Σᵢ Dᵢ": round(float(d.sum()), 3),
            "loss": round(float(energy["loss"][t, 0]), 4),
        }
    )
pd.DataFrame(rows)

Read the table from top to bottom. Early on every feature has a small Dᵢ (random directions
in 30 dimensions). After a few thousand steps almost everything has settled: most features at
0, the rest at 1/2. But Σᵢ Dᵢ stays a little below m = 30: one dimension is **not** a clean
digon; a few weak features share it. The model then sits on a plateau for tens of thousands
of steps until two of those features win the dimension and jump to 1/2, and the loss drops.
That is the paper's picture: "seemingly smooth decreases of the loss curve in larger models
are in fact composed of many small jumps of features between different configurations".

Every jump the detector found, with the loss around it:

In [ ]:
run.read_table("jumps")

## 2. Learning as geometric transformations (p. 27)

Setup: n = 6 features in m = 3 dimensions, as **two correlated sets** of three (features in a
set are zero or active together, p. 23). Weights start close to zero (0.1 × the usual init,
as the paper describes). The paper's trajectory: (A) near zero → (B) the two sets push apart →
(C) each set expands into a triangle → (D) the triangles rotate into an antiprism, an
octahedron.

**Predict:** in the final octahedron, which features form antipodal pairs: two features of the
same set, or one from each set? Why? (Think about what happens to an antipodal pair when both
features are active at once.)

In [ ]:
display(Image(filename=str(run.figures_dir / "trajectories.png")))
display(Image(filename=str(run.figures_dir / "stage_raster.png")))

In [ ]:
from superposition.experiments.learning_dynamics import COLLAPSED

trans = run.load_checkpoint("transformations")
table_b = run.read_table("metrics")
sets = np.asarray(trans["sets"])
spacing = int(trans["steps"][1] - trans["steps"][0])
rows = []
for p, group in table_b.groupby("set_probability"):
    durations, centroid_cos = [], []
    for k in group["instance"]:
        collapsed = np.flatnonzero(trans["stages"][k] == COLLAPSED)
        durations.append(len(collapsed) * spacing)
        for t in collapsed:
            W = trans["W"][t, k].numpy()
            a, b = W[:, sets == 0].mean(axis=1), W[:, sets == 1].mean(axis=1)
            centroid_cos.append(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
    rows.append(
        {
            "1 − S": p,
            "runs visiting 'collapsed'": f"{sum(d > 0 for d in durations)}/{len(durations)}",
            "median steps collapsed": int(np.median(durations)),
            "median cos(set A, set B) while collapsed": round(float(np.median(centroid_cos)), 2),
        }
    )
pd.DataFrame(rows)

Every run passes through the collapsed stage, and while it is there the two sets point in
nearly orthogonal directions: the linear two-component solution (one direction per set, a
PCA-like answer). The paper's panel B looks like the sets pushed to opposite ends of one axis;
in our runs they only become opposed later, in the prism stage (docs/REPRODUCTION.md, E05).

The raster shows every restart (rows) over training (log steps), colored by its stage:
**collapsed** = each set moves as one direction (the linear, PCA-like solution of stage B);
**prism** = two triangles on opposite sides, aligned (stage C); **octahedron** = antipodal
pairs across sets (stage D). Drag the 3D view below to see one run's trajectories.

In [ ]:
from superposition.experiments.learning_dynamics import representative_restart
from superposition.plotting.dynamics import trajectories_3d

dyn = lab.load_dynamics(run)
k = representative_restart(dyn.table, float(dyn.table["init_scale"].iloc[0]))
trajectories_3d(lab.trajectory_in_frame(dyn, k), dyn.trans_steps, dyn.sets)

In [ ]:
dyn.table.groupby(["set_probability", "trajectory_type"]).size().rename("restarts").to_frame()

Answer to the prediction: antipodal pairs always join features from **different** sets.
Features of the same set are active together, and an antipodal pair of co-active features
cancels: each one's positive value pushes the other's output negative. Features of different
sets are rarely active together, so the ReLU removes their (negative) interference for free.

## Exercises

**1.** For the restart drawn in the energy-level figure, find the step of its detected jump
and compute Σᵢ Dᵢ just before and just after it. How much of the "missing" dimension does the
jump recover?

<details><summary>Solution</summary>

```python
jumps = run.read_table("jumps")
j = jumps[jumps["coincides_with_drop"]].sort_values("loss_drop").iloc[-1]
r = int(j["restart"])
before = int(np.searchsorted(steps, j["start_step"] - 5_000))
after = int(np.searchsorted(steps, j["end_step"] + 5_000))
print(dims[before, r].sum(), dims[min(after, len(steps) - 1), r].sum())
```

Σᵢ Dᵢ goes from about 29.9 to about 30.0: the jump turns the shared, badly used 30th
dimension into one more clean digon, and the loss drops by roughly the value of storing two
more features.
</details>

**2.** Why does the detector require the feature to sit at one level for 5,000 steps before
it jumps? What would it report without that condition?

<details><summary>Solution</summary>

Without it, the initial organization of the weights (every Dᵢ moving from ~0.1 to 0 or 1/2 in
the first few thousand steps, while the loss falls steeply) would count as hundreds of
"jumps" that trivially coincide with a falling loss. The paper's phenomenon is a jump
*between* levels on a plateau, which is what the stable-window condition selects.
</details>

**3.** Compare the trajectory types at the three set densities. At which density do runs
spend longest in the aligned prism stage before rotating into the octahedron? Relate this to
the paper's remark that "the different trajectories vary at step C".

<details><summary>Solution</summary>

```python
dyn.table.groupby("set_probability")[["prism_steps", "first_octahedron_step"]].median()
```

In the cited quick run every restart ends as an octahedron, but the routes differ. At
1 − S = 0.01 the median run sits in the aligned prism for about 4,000 steps; at 0.1 about
3,300; at 0.03 only about 1,500, and several runs there skip the prism altogether (types
"transition → octahedron"). The rotation from prism to antiprism is driven only by inputs in
which both sets are active, which are rare when sets are sparse, so a long prism plateau is
plausible at 1 − S = 0.01. Three densities and 16 runs each are too few to claim a trend; the
point is that step C varies from run to run, as the paper says.
</details>

**4.** The paper says weights start "randomly close to zero". Re-run part (b) with the usual
Xavier initialization and compare the trajectory types.

<details><summary>Solution</summary>

```python
xavier = run_experiment(
    "learning_dynamics", "quick", device="auto",
    overrides={"energy": {"n_restarts": 1}, "train": {"steps": 2_000},
               "transformations": {"init_scales": [1.0]}},
)
lab.load_dynamics(xavier).table.groupby(["set_probability", "trajectory_type"]).size()
```

(The overrides shorten part (a), which this exercise does not need.) With Xavier init the
feature norms start near 0.75, so no run passes through the "small" stage (A). Whether runs
still collapse each set onto one direction (B) before expanding, and whether they still end as
octahedra, is what you are checking: compare the counts with the table above. The full profile
runs both inits side by side.
</details>